# FIXED COPY (DDMpy_log)

This notebook is a copy of `Fervo modeling with tensile and shear 07_2026_claude.ipynb`, modified to use the parallel `DDMpy_log` package instead of `DDMpy` (which is left untouched).

**Root cause found:** `data.take_time_diff()` (in JIN_pylib's `Data2D_XT.py`) divides by `np.diff(self.taxis)` where `taxis` is in **minutes**, but downstream code treats the result as strain **per second** (e.g. multiplying by 60 to get `dt_seconds` for the trapezoidal reintegration, or by `1e9` to label it nanostrain/s). That unit mismatch inflates the reconstructed/plotted strain rate by ~60x, and was the dominant source of the ~1.3e-4 direct-vs-integrated strain discrepancy (a much smaller, secondary effect: reintegrating a *backward*-difference rate with the trapezoidal rule also introduces a systematic half-step lag).

**Fix (`DDMpy_log/StrainRateUtil.py`):** `centered_time_diff` and `integrate_rate_trapezoidal` both take the *same* explicit time axis, so the two operations can never end up in mismatched units. Centered differencing also removes the half-step lag. Cells 10 & 13 additionally use `StrainRateUtil.build_dense_taxis` to densify sampling only around the T1/T2 transitions (not uniformly), keeping the added DDM evaluation cost small since each sample is a closed-form elasticity evaluation (no PDE solve).

Validated on the exact fault-history setup from cell 10: max direct-vs-integrated discrepancy dropped from `1.285e-04` (original, buggy) to `~4e-10` (fixed), with DDM evaluation time going from ~1.9s to ~6.7s for the densified case (well within budget).


In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
%pip install plotly
%pip install ipympl  
%matplotlib widget


In [ ]:
import sys
import os

# sys.path.append(os.path.abspath("D:\OneDrive - Colorado School of Mines\Research")) # DDMPy V2
sys.path.append(os.path.abspath("D:\OneDrive\文档\GitHub")) # DDMPy V3

print(sys.path)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from DDMpy_log import *
from DDMpy_log import StrainRateUtil

so the code below is to simulate tensile to shear
2 issues need to be done
first is that how to make up the gap? Jin did this
second is that I need put the fracture center exactly at the perforation point
third is to add the tensile with the remote heart shape
which are a lot of pieces of work to do

# History match test 1

In [ ]:
# This code simulate Fervo's Golden 4 monitoring well response to a preexisting fault rupture 
# The fault acts like a tensile fracture in the first several days, then it becomes a shear fault

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from datetime import datetime, timedelta
import os
import copy

# load control points from a csv file
base_path = r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Well Geometries"
df_gold = pd.read_csv(f"{base_path}\\Gold_4_PB_Well_Geometry.csv")
control_points_G4 = df_gold[['x_gold', 'y_gold', 'z_gold']].values
tvd = df_gold['TVDrkb'].values
md = df_gold['MD'].values
# control_points_G4 = np.array([[-3000,0,0],
#                             [-3000,0,-10000]
#                             ,[3000,0,-10000]])
#well_elem_N = 100  # Number of elements in the well
well_elem_N = len(control_points_G4)*10

# smooth_val = well_elem_N // 10 
smooth_val = 30
smooth = smooth_val if smooth_val % 2 == 1 else smooth_val + 1
well = Well.set_well_by_points(control_points_G4, N=well_elem_N, smooth=smooth)
well.gauge_length = 30.48  # gauge length in ft (10m)

#%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

############################################################
# Define the fault
strike = 0  # strike angle in degrees
dip = 90  # dip angle in degrees
# L = 2800
L = 4000 # length of the fault in ft. this can be ajusted, but need to ensure it hits the monitoring well
# L = 3200
W_max = 10 / 1000 / 5 / 10 / 10 * 60 # max width (ft)
S1_max = -10 / 1000 * 2 / 10 * 60 # max shear displacement(ft)

# x is constrained by the hitting channel of the monitoring well Golden 4
# which can be read from the LFDAS waterfall plot
# also which has a range from md 10320-10390 approximately
# so here x is set to be corresponding to md of 10340
df_xyz = pd.read_csv(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\input\pumping_windows_stage_with_coordinates_xyz.csv")
x = df_xyz['X'].values[5]
y = df_xyz['Y'].values[5]
z = df_xyz['Z'].values[5]

frac = DynamicFracture.GlobalRectangularFracture()
frac.set_global_coors(strike, dip, x, y, z)
####################################################

####################################################
# Define the fault behavior
# The fault behaves like a tensile fracture in the first several days, then it becomes a shear fault

# Time ranges(converted to minutes since 00:00 2025-02-24)
T0 = datetime(2025, 2, 24, 0, 0)
T1 = datetime(2025, 2, 24, 11, 0)
T2 = datetime(2025, 2, 28, 0, 0)
T3 = datetime(2025, 3, 3, 22, 0)

def mins(dt):
    return (dt - T0).total_seconds() / 60

t1 = mins(T1)
t2 = mins(T2)
t3 = mins(T3)

N = 144

delta_t = t3 / N  # total time divided by number of points

N1, N2, N3 = int(t1 // delta_t), int((t2-t1) // delta_t + 1), int((t3-t2+1) // delta_t + 1)
t_stage1 = np.linspace(0, t1 - delta_t, N1)
t_stage2 = np.linspace(t1, t2 - delta_t, N2)
t_stage3 = np.linspace(t2, t3, N3)

taxis = np.concatenate([t_stage1, t_stage2, t_stage3])

length = np.full_like(taxis, L)

# === Stage 1: no displacement
width_stage1 = np.zeros(N1)
#epsilon = 1e-6  # small value to avoid division by zero
#width_ratio = np.linspace(0, epsilon, N1)  # epsilon is a small value to avoid division by zero
shear_stage1 = np.zeros(N1)  
# === Stage 2: growing tensile (width only)
t_norm = np.linspace(0, 1, N2)
width_stage2 = W_max * (t_norm ** 2)
shear_stage2 = np.zeros(N2)
# === Stage 3: two options
# constant width, growing shear 
option = 'A' #choose: "A" = growing width +shear; "B" = fix width, growing shear
if option == "A":
    width_stage3 = np.linspace(W_max, W_max*5, N3+1)
    width_stage3 = width_stage3[1:]
else: # option B
    width_stage3 = np.full(N3, W_max)

shear_stage3 = np.linspace(0, S1_max, N3)

# combine all stages
width = np.concatenate([width_stage1, width_stage2, width_stage3])
S1 = np.concatenate([shear_stage1, shear_stage2, shear_stage3])
S2 = np.zeros_like(S1)

frac.define_LHW(
    taxis = taxis,
    length = length,
    height = np.ones_like(length)*5000,
    # width_ratio = width_ratio,
    width = width,  # fixed width ratio for the whole time
    #S1_ratio = S1_ratio,
    S1 = S1,  # fixed S1 ratio for the whole time
    S2 = None,
)

frac.set_monitor_wells(well)
frac.calculate()

# === 3. Extract Strain and Calculate Strain Rate ===
# Keep strain and strain rate in separate, clearly named objects.
strain_data = frac.gather_strain_data()[0]
strain_rate_data = copy.deepcopy(strain_data)
strain_rate_data.data = StrainRateUtil.centered_time_diff(
    np.asarray(strain_data.data, dtype=float),
    np.asarray(strain_data.taxis, dtype=float) * 60.0,
)  # per-second rate; units-consistent fix (was: data.take_time_diff(), which divides
   # by np.diff(taxis) in MINUTES while downstream code assumes strain/second)


# === 4. Plot using plot_waterfall ===
# 
plt.figure(figsize=(10, 6))
strain_rate_data.plot_waterfall()
ax = plt.gca()

# 设置 tick 时间点（单位仍是分钟）
start_time = datetime(2025, 2, 24, 0, 0)
end_time = datetime(2025, 3, 4, 0, 0)
total_minutes = (end_time - start_time).total_seconds() / 60
tick_minutes = np.arange(0, total_minutes + 1, 24 * 60)
tick_labels = [
    (start_time + timedelta(minutes=int(m))).strftime('%m/%d\n%H:%M')
    for m in tick_minutes
]

# 设置 tick 显示
ax.set_xticks(tick_minutes)
ax.set_xticklabels(tick_labels, rotation=0, ha='center')

# Fixed strain-rate color range: [-0.3, 0.3] nanostrain/s.
ax.images[0].set_clim(-0.3e-9, 0.3e-9)
rate_cbar = plt.colorbar(ax.images[0], ax=ax)
rate_cbar.set_label('Strain rate (nanostrain/s)')
rate_cbar.formatter = FuncFormatter(lambda value, _: f'{value * 1e9:g}')
rate_cbar.update_ticks()

# plot vertical curve at T1, T2, T3
ax.axvline(x=t1, color='green', linestyle='--', label='T1')
ax.axvline(x=t2, color='green', linestyle='--', label='T2')
ax.axvline(x=t3, color='green', linestyle='--', label='T3')

# 其余设置
ax.set_xlabel('Time')
ax.set_ylabel('Measured Depth (ft)')
ax.set_ylim([14500, 8000])
plt.tight_layout()
plt.show()


# === 5. Fracture Drawing and Save ===
maxS1 = np.max([np.abs(e.S1) for e in frac.fractures[-1].elements])
maxS2 = np.max([np.abs(e.S2) for e in frac.fractures[-1].elements])
maxwidth = np.max([np.abs(e.width) for e in frac.fractures[-1].elements])
max_color = np.max([maxS1, maxS2, maxwidth])
max_color = 0.02  # add some buffer

fig = frac.draw(tensile=max_color)
# plt.title(f"Strain rate Response\nWidth: {width_ratio}, Horizontal Shear Slip: {S1_ratio}, Dip Slip: {S2_ratio}")

fig.show()


In [ ]:
# === 5. Fracture Drawing and Save ===
maxS1 = np.max([np.abs(e.S1) for e in frac.fractures[-1].elements])
maxS2 = np.max([np.abs(e.S2) for e in frac.fractures[-1].elements])
maxwidth = np.max([np.abs(e.width) for e in frac.fractures[-1].elements])
max_color = np.max([maxS1, maxS2, maxwidth])
max_color = 0.3  # add some buffer

# fig = frac.draw(tensile=max_color*100)
fig = frac.draw()
#  plt.title(f"Strain rate Response\nWidth: {width_ratio}, Horizontal Shear Slip: {S1_ratio}, Dip Slip: {S2_ratio}")

fig.show()

# test the fault length

In [ ]:
# This code simulate Fervo's Golden 4 monitoring well response to a preexisting fault rupture 
# The fault acts like a tensile fracture in the first several days, then it becomes a shear fault

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from datetime import datetime, timedelta
import os
import copy

# load control points from a csv file
base_path = r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Well Geometries"
df_gold = pd.read_csv(f"{base_path}\\Gold_4_PB_Well_Geometry.csv")
control_points_G4 = df_gold[['x_gold', 'y_gold', 'z_gold']].values
tvd = df_gold['TVDrkb'].values
md = df_gold['MD'].values
# control_points_G4 = np.array([[-3000,0,0],
#                             [-3000,0,-10000]
#                             ,[3000,0,-10000]])
#well_elem_N = 100  # Number of elements in the well
well_elem_N = len(control_points_G4)*10

# smooth_val = well_elem_N // 10 
smooth_val = 30
smooth = smooth_val if smooth_val % 2 == 1 else smooth_val + 1
well = Well.set_well_by_points(control_points_G4, N=well_elem_N, smooth=smooth)
well.gauge_length = 30.48  # gauge length in ft (10m)

#%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

############################################################
# Define the fault
strike = -0.8  # strike angle in degrees
dip = 90  # dip angle in degrees
# L = 2800
L = 3900 # length of the fault in ft. this can be ajusted, but need to ensure it hits the monitoring well
# L = 3200
W_max = 10 / 1000 / 5 / 10 / 10 / 2 * 60  # max width (ft)
S1_max = -10 / 1000 * 2 / 10 / 2 * 60 # max shear displacement(ft)

# x is constrained by the hitting channel of the monitoring well Golden 4
# which can be read from the LFDAS waterfall plot
# also which has a range from md 10320-10390 approximately
# so here x is set to be corresponding to md of 10340
df_xyz = pd.read_csv(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\input\pumping_windows_stage_with_coordinates_xyz.csv")
x = df_xyz['X'].values[5]
y = df_xyz['Y'].values[5]
z = df_xyz['Z'].values[5]

frac = DynamicFracture.GlobalRectangularFracture()
frac.set_global_coors(strike, dip, x, y, z)
####################################################

####################################################
# Define the fault behavior
# The fault behaves like a tensile fracture in the first several days, then it becomes a shear fault

# Time ranges(converted to minutes since 00:00 2025-02-24)
T0 = datetime(2025, 2, 24, 0, 0)
T1 = datetime(2025, 2, 24, 11, 0)
T2 = datetime(2025, 2, 28, 0, 0)
T3 = datetime(2025, 3, 3, 22, 0)

def mins(dt):
    return (dt - T0).total_seconds() / 60

t1 = mins(T1)
t2 = mins(T2)
t3 = mins(T3)

N = 144

delta_t = t3 / N  # total time divided by number of points

N1, N2, N3 = int(t1 // delta_t), int((t2-t1) // delta_t + 1), int((t3-t2+1) // delta_t + 1)
t_stage1 = np.linspace(0, t1 - delta_t, N1)
t_stage2 = np.linspace(t1, t2 - delta_t, N2)
t_stage3 = np.linspace(t2, t3, N3)

taxis = np.concatenate([t_stage1, t_stage2, t_stage3])

length = np.full_like(taxis, L)

# === Stage 1: no displacement
width_stage1 = np.zeros(N1)
#epsilon = 1e-6  # small value to avoid division by zero
#width_ratio = np.linspace(0, epsilon, N1)  # epsilon is a small value to avoid division by zero
shear_stage1 = np.zeros(N1)  

# === Stage 2: growing tensile (width only)
t_norm = np.linspace(0, 1, N2)
width_stage2 = W_max * (t_norm ** 2)
shear_stage2 = np.zeros(N2)

# === Stage 3: two options
# constant width, growing shear 
option = 'A' #choose: "A" = growing width +shear; "B" = fix width, growing shear
if option == "A":
    width_stage3 = np.linspace(W_max, W_max*5, N3+1)
    width_stage3 = width_stage3[1:]
else: # option B
    width_stage3 = np.full(N3, W_max)

shear_stage3 = np.linspace(0, S1_max, N3)

# combine all stages
width = np.concatenate([width_stage1, width_stage2, width_stage3])
S1 = np.concatenate([shear_stage1, shear_stage2, shear_stage3])
S2 = np.zeros_like(S1)

frac.define_LHW(
    taxis = taxis,
    length = length,
    height = np.ones_like(length)*5000,
    # width_ratio = width_ratio,
    width = width,  # fixed width ratio for the whole time
    #S1_ratio = S1_ratio,
    S1 = S1,  # fixed S1 ratio for the whole time
    S2 = None,
)

frac.set_monitor_wells(well)
frac.calculate()

# === 3. Extract Strain and Calculate Strain Rate ===
# Follow the same strain/strain-rate pattern used in "History match test 1".
strain_data = frac.gather_strain_data()[0]
strain_rate_data = copy.deepcopy(strain_data)
strain_rate_data.data = StrainRateUtil.centered_time_diff(
    np.asarray(strain_data.data, dtype=float),
    np.asarray(strain_data.taxis, dtype=float) * 60.0,
)  # per-second rate; units-consistent fix (was: data.take_time_diff(), which divides
   # by np.diff(taxis) in MINUTES while downstream code assumes strain/second)


# === 4. Shared time ticks for waterfall plots ===
start_time = datetime(2025, 2, 24, 0, 0)
end_time = datetime(2025, 3, 4, 0, 0)
total_minutes = (end_time - start_time).total_seconds() / 60
tick_minutes = np.arange(0, total_minutes + 1, 24 * 60)
tick_labels = [
    (start_time + timedelta(minutes=int(m))).strftime('%m/%d\n%H:%M')
    for m in tick_minutes
]

# === 5. Fracture Drawing and Save ===
maxS1 = np.max([np.abs(e.S1) for e in frac.fractures[-1].elements])
maxS2 = np.max([np.abs(e.S2) for e in frac.fractures[-1].elements])
maxwidth = np.max([np.abs(e.width) for e in frac.fractures[-1].elements])
max_color = np.max([maxS1, maxS2, maxwidth])
max_color = 0.02  # add some buffer

fig = frac.draw(tensile=max_color)
# plt.title(f"Strain rate Response\nWidth: {width_ratio}, Horizontal Shear Slip: {S1_ratio}, Dip Slip: {S2_ratio}")

fig.show()

# plot strain-rate and T1-referenced strain waterfalls together for comparison
# The strain reference profile is the simulated strain profile at the time sample nearest T1.
strain_t1_ref_data = copy.deepcopy(strain_data)
t1_strain_idx = int(np.argmin(np.abs(np.asarray(strain_t1_ref_data.taxis, dtype=float) - t1)))
t1_strain_time = strain_t1_ref_data.taxis[t1_strain_idx]
strain_t1_ref_data.data = strain_t1_ref_data.data - strain_t1_ref_data.data[:, [t1_strain_idx]]

fig, (ax_rate, ax_strain) = plt.subplots(2, 1, figsize=(11, 9), sharex=True, constrained_layout=True)

plt.sca(ax_rate)
strain_rate_data.plot_waterfall()
rate_mappable = ax_rate.images[-1] if ax_rate.images else ax_rate.collections[-1]
rate_color_abs = 0.3e-9
rate_mappable.set_clim(-rate_color_abs, rate_color_abs)
rate_cbar = fig.colorbar(rate_mappable, ax=ax_rate)
rate_cbar.set_label('Strain rate (nanostrain/s)')
rate_cbar.formatter = FuncFormatter(lambda value, _: f'{value * 1e9:g}')
rate_cbar.update_ticks()
ax_rate.set_title('Simulated strain-rate waterfall at Golden 4-PB')

plt.sca(ax_strain)
strain_t1_ref_data.plot_waterfall()
strain_mappable = ax_strain.images[-1] if ax_strain.images else ax_strain.collections[-1]
strain_color_abs = 0.1e-3  # 0.1 millistrain in dimensionless strain units
strain_mappable.set_clim(-strain_color_abs, strain_color_abs)
strain_cbar = fig.colorbar(strain_mappable, ax=ax_strain)
strain_cbar.set_label('T1-referenced strain (millistrain)')
strain_cbar.formatter = FuncFormatter(lambda value, _: f'{value * 1e3:g}')
strain_cbar.update_ticks()
ax_strain.set_title(f'Simulated T1-referenced strain waterfall at Golden 4-PB; reference sample: {t1_strain_time:.1f} min')

for ax in (ax_rate, ax_strain):
    ax.set_ylabel('Measured Depth (ft)')
    ax.set_ylim([10500, 10200])
    ax.set_xlim([t1, t3])
    ax.set_xticks(tick_minutes)
    ax.set_xticklabels(tick_labels, rotation=0, ha='center')
    ax.axvline(x=t1, color='green', linestyle='--', label='T1')
    ax.axvline(x=t2, color='green', linestyle='--', label='T2')
    ax.axvline(x=t3, color='green', linestyle='--', label='T3')
    ax.legend(loc='best')

ax_strain.set_xlabel('Time')
plt.show()



# test the fault length 2

In [ ]:
# This code simulate Fervo's Golden 4 monitoring well response to a preexisting fault rupture 
# The fault acts like a tensile fracture in the first several days, then it becomes a shear fault

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from datetime import datetime, timedelta
import os
import copy

# load control points from a csv file
base_path = r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Well Geometries"
df_gold = pd.read_csv(f"{base_path}\\Gold_4_PB_Well_Geometry.csv")
control_points_G4 = df_gold[['x_gold', 'y_gold', 'z_gold']].values
tvd = df_gold['TVDrkb'].values
md = df_gold['MD'].values
# control_points_G4 = np.array([[-3000,0,0],
#                             [-3000,0,-10000]
#                             ,[3000,0,-10000]])
#well_elem_N = 100  # Number of elements in the well
well_elem_N = len(control_points_G4)*10

# smooth_val = well_elem_N // 10 
smooth_val = 30
smooth = smooth_val if smooth_val % 2 == 1 else smooth_val + 1
well = Well.set_well_by_points(control_points_G4, N=well_elem_N, smooth=smooth)
well.gauge_length = 30.48  # gauge length in ft (10m)

#%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

############################################################
# Define the fault
strike = -0.8  # strike angle in degrees
dip = 90  # dip angle in degrees
# L = 2800
L = 3900 # length of the fault in ft. this can be ajusted, but need to ensure it hits the monitoring well
# L = 3200
W_max = 10 / 1000 / 5 / 10 / 10 / 2 * 60  # max width (ft)
S1_max = -10 / 1000 * 2 / 10 / 2 * 60 # max shear displacement(ft)

# x is constrained by the hitting channel of the monitoring well Golden 4
# which can be read from the LFDAS waterfall plot
# also which has a range from md 10320-10390 approximately
# so here x is set to be corresponding to md of 10340
df_xyz = pd.read_csv(r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\input\pumping_windows_stage_with_coordinates_xyz.csv")
x = df_xyz['X'].values[5]
y = df_xyz['Y'].values[5]
z = df_xyz['Z'].values[5]

frac = DynamicFracture.GlobalRectangularFracture()
frac.set_global_coors(strike, dip, x, y, z)
####################################################

####################################################
# Define the fault behavior
# The fault behaves like a tensile fracture in the first several days, then it becomes a shear fault

# Time ranges(converted to minutes since 00:00 2025-02-24)
T0 = datetime(2025, 2, 24, 0, 0)
T1 = datetime(2025, 2, 24, 11, 0)
T2 = datetime(2025, 2, 28, 0, 0)
T3 = datetime(2025, 3, 3, 22, 0)

def mins(dt):
    return (dt - T0).total_seconds() / 60

t1 = mins(T1)
t2 = mins(T2)
t3 = mins(T3)

N = 144

delta_t = t3 / N  # total time divided by number of points

N1, N2, N3 = int(t1 // delta_t), int((t2-t1) // delta_t + 1), int((t3-t2+1) // delta_t + 1)
t_stage1 = np.linspace(0, t1 - delta_t, N1)
t_stage2 = np.linspace(t1, t2 - delta_t, N2)
t_stage3 = np.linspace(t2, t3, N3)

taxis = np.concatenate([t_stage1, t_stage2, t_stage3])

length = np.full_like(taxis, L)

# === Stage 1: no displacement
width_stage1 = np.zeros(N1)
#epsilon = 1e-6  # small value to avoid division by zero
#width_ratio = np.linspace(0, epsilon, N1)  # epsilon is a small value to avoid division by zero
shear_stage1 = np.zeros(N1)  

# === Stage 2: growing tensile (width only)
t_norm = np.linspace(0, 1, N2)
width_stage2 = W_max * (t_norm ** 2)
shear_stage2 = np.zeros(N2)

# === Stage 3: two options
# constant width, growing shear 
option = 'A' #choose: "A" = growing width +shear; "B" = fix width, growing shear
if option == "A":
    width_stage3 = np.linspace(W_max, W_max*5, N3+1)
    width_stage3 = width_stage3[1:]
else: # option B
    width_stage3 = np.full(N3, W_max)

shear_stage3 = np.linspace(0, S1_max, N3)

# combine all stages
width = np.concatenate([width_stage1, width_stage2, width_stage3])
S1 = np.concatenate([shear_stage1, shear_stage2, shear_stage3])
S2 = np.zeros_like(S1)

frac.define_LHW(
    taxis = taxis,
    length = length,
    height = np.ones_like(length)*5000,
    # width_ratio = width_ratio,
    width = width,  # fixed width ratio for the whole time
    #S1_ratio = S1_ratio,
    S1 = S1,  # fixed S1 ratio for the whole time
    S2 = None,
)

frac.set_monitor_wells(well)
frac.calculate()

# === 3. Extract direct strain, calculate strain rate, and integrate it back to strain ===
# Direct DDM strain and strain rate remain separate objects.
strain_data = frac.gather_strain_data()[0]
taxis_seconds = np.asarray(strain_data.taxis, dtype=float) * 60.0

strain_rate_data = copy.deepcopy(strain_data)
strain_rate_data.data = StrainRateUtil.centered_time_diff(
    np.asarray(strain_data.data, dtype=float),
    taxis_seconds,
)

# Reference direct DDM strain to the sample exactly at/nearest T1.
strain_t1_ref_data = copy.deepcopy(strain_data)
t1_strain_idx = int(np.argmin(np.abs(np.asarray(strain_t1_ref_data.taxis, dtype=float) - t1)))
t1_strain_time = float(strain_t1_ref_data.taxis[t1_strain_idx])
strain_t1_ref_data.data = (
    np.asarray(strain_t1_ref_data.data, dtype=float)
    - np.asarray(strain_t1_ref_data.data, dtype=float)[:, [t1_strain_idx]]
)

# Integrate the strain rate with the same seconds-based time axis used above.
integrated_strain_data = copy.deepcopy(strain_rate_data)
integrated_strain_data.data = StrainRateUtil.integrate_rate_trapezoidal(
    np.asarray(strain_rate_data.data, dtype=float),
    taxis_seconds,
)

# Apply the same T1 reference to the integrated strain for a direct comparison.
integrated_strain_t1_ref_data = copy.deepcopy(integrated_strain_data)
t1_integrated_idx = int(
    np.argmin(np.abs(np.asarray(integrated_strain_t1_ref_data.taxis, dtype=float) - t1))
)
t1_integrated_time = float(integrated_strain_t1_ref_data.taxis[t1_integrated_idx])
integrated_strain_t1_ref_data.data = (
    np.asarray(integrated_strain_t1_ref_data.data, dtype=float)
    - np.asarray(integrated_strain_t1_ref_data.data, dtype=float)[:, [t1_integrated_idx]]
)

strain_integration_difference = (
    np.asarray(strain_t1_ref_data.data, dtype=float)
    - np.asarray(integrated_strain_t1_ref_data.data, dtype=float)
)
max_abs_strain_integration_difference = float(
    np.nanmax(np.abs(strain_integration_difference))
)


# === 4. Shared time ticks for waterfall plots ===
start_time = datetime(2025, 2, 24, 0, 0)
end_time = datetime(2025, 3, 4, 0, 0)
total_minutes = (end_time - start_time).total_seconds() / 60
tick_minutes = np.arange(0, total_minutes + 1, 24 * 60)
tick_labels = [
    (start_time + timedelta(minutes=int(m))).strftime('%m/%d\n%H:%M')
    for m in tick_minutes
]

# === 5. Fracture Drawing and Save ===
maxS1 = np.max([np.abs(e.S1) for e in frac.fractures[-1].elements])
maxS2 = np.max([np.abs(e.S2) for e in frac.fractures[-1].elements])
maxwidth = np.max([np.abs(e.width) for e in frac.fractures[-1].elements])
max_color = np.max([maxS1, maxS2, maxwidth])
max_color = 0.02  # add some buffer

fig = frac.draw(tensile=max_color)
# plt.title(f"Strain rate Response\nWidth: {width_ratio}, Horizontal Shear Slip: {S1_ratio}, Dip Slip: {S2_ratio}")

fig.show()

# Plot strain rate, direct DDM strain, and strain integrated from strain rate.
fig, (ax_rate, ax_strain_direct, ax_strain_integrated) = plt.subplots(
    3,
    1,
    figsize=(11, 12),
    sharex=True,
    constrained_layout=True,
)

plt.sca(ax_rate)
strain_rate_data.plot_waterfall()
rate_mappable = ax_rate.images[-1] if ax_rate.images else ax_rate.collections[-1]
rate_mappable.set_clim(-0.3e-9, 0.3e-9)
rate_cbar = fig.colorbar(rate_mappable, ax=ax_rate)
rate_cbar.set_label('Strain rate (nanostrain/s)')
rate_cbar.formatter = FuncFormatter(lambda value, _: f'{value * 1e9:g}')
rate_cbar.update_ticks()
ax_rate.set_title('Simulated strain-rate waterfall at Golden 4-PB')

plt.sca(ax_strain_direct)
strain_t1_ref_data.plot_waterfall()
direct_strain_mappable = (
    ax_strain_direct.images[-1]
    if ax_strain_direct.images
    else ax_strain_direct.collections[-1]
)
direct_strain_mappable.set_clim(-0.1e-3, 0.1e-3)
direct_strain_cbar = fig.colorbar(direct_strain_mappable, ax=ax_strain_direct)
direct_strain_cbar.set_label('Direct DDM strain (millistrain)')
direct_strain_cbar.formatter = FuncFormatter(lambda value, _: f'{value * 1e3:g}')
direct_strain_cbar.update_ticks()
ax_strain_direct.set_title(
    f'Direct DDM T1-referenced strain; reference sample: {t1_strain_time:.1f} min'
)

plt.sca(ax_strain_integrated)
integrated_strain_t1_ref_data.plot_waterfall()
integrated_strain_mappable = (
    ax_strain_integrated.images[-1]
    if ax_strain_integrated.images
    else ax_strain_integrated.collections[-1]
)
integrated_strain_mappable.set_clim(-0.1e-3, 0.1e-3)
integrated_strain_cbar = fig.colorbar(
    integrated_strain_mappable,
    ax=ax_strain_integrated,
)
integrated_strain_cbar.set_label('Strain integrated from strain rate (millistrain)')
integrated_strain_cbar.formatter = FuncFormatter(lambda value, _: f'{value * 1e3:g}')
integrated_strain_cbar.update_ticks()
ax_strain_integrated.set_title(
    'T1-referenced strain integrated from strain rate; '
    f'reference sample: {t1_integrated_time:.1f} min'
)

for ax in (ax_rate, ax_strain_direct, ax_strain_integrated):
    ax.set_ylabel('Measured Depth (ft)')
    ax.set_ylim([10500, 10200])
    ax.set_xlim([t1, t3])
    ax.set_xticks(tick_minutes)
    ax.set_xticklabels(tick_labels, rotation=0, ha='center')
    ax.axvline(x=t1, color='green', linestyle='--', label='T1')
    ax.axvline(x=t2, color='green', linestyle='--', label='T2')
    ax.axvline(x=t3, color='green', linestyle='--', label='T3')
    ax.legend(loc='best')

ax_strain_integrated.set_xlabel('Time')
plt.show()

print(f'Direct strain T1 reference sample: {t1_strain_time:.1f} min')
print(f'Integrated strain T1 reference sample: {t1_integrated_time:.1f} min')
print(
    'Maximum absolute direct-vs-integrated T1-referenced strain difference: '
    f'{max_abs_strain_integration_difference:.6e}'
)



## plot and output the 4 hour mean strain rate and strain profile.

In [ ]:
# Direct-DDM strain-rate and strain waterfalls with observation-style 4-hour profiles
# Run this cell immediately after "test the fault length 2".
# The strain waterfall/profile uses direct DDM strain, not integrated strain.

from pathlib import Path
import matplotlib.dates as mdates

sim_plot_start = pd.Timestamp(T1)
sim_plot_end = pd.Timestamp(T3)
sim_md_top = 10200
sim_md_bottom = 10500
sim_profile_interval = "4h"
sim_rate_overlay_half_width_hours = 28.0
sim_strain_overlay_half_width_hours = 28.0
sim_profile_scale_multiplier = 10.0
sim_rate_color_scale = (-0.3, 0.3)      # nanostrain/s
sim_strain_color_scale = (-0.1, 0.1)    # millistrain

MODEL_RATE_TO_NANOSTRAIN_PER_S = 1e9
MODEL_STRAIN_TO_MILLISTRAIN = 1e3

sim_export_dir = Path(
    r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports\sim_fault2"
)
sim_export_dir.mkdir(parents=True, exist_ok=True)
sim_export_suffix = (
    f"{sim_plot_start:%Y%m%d_%H%M}_to_{sim_plot_end:%Y%m%d_%H%M}_"
    f"{int(sim_md_top)}_{int(sim_md_bottom)}ft_{sim_profile_interval}_mean_T1_ref"
)

sim_time_marks = pd.DatetimeIndex([T1, T2, T3])
sim_mark_labels = ["T1", "T2", "T3"]


def get_ddm_depth_axis(data_obj):
    """Return the measured-depth coordinate stored on a DDM data object."""
    for name in ("daxis", "depth", "md", "yaxis", "xaxis"):
        if hasattr(data_obj, name):
            values = getattr(data_obj, name)
            if values is not None:
                return np.asarray(values, dtype=float)
    return np.arange(np.asarray(data_obj.data).shape[0], dtype=float)


def compute_mean_profiles(data_matrix, time_index, start_time, end_time, interval):
    """Compute forward-window means and anchor each plotted profile at its window start."""
    time_index = pd.DatetimeIndex(time_index)
    window_starts = pd.date_range(start_time, end_time, freq=interval)
    profiles = []
    rows = []

    for window_start in window_starts:
        window_end = min(window_start + pd.Timedelta(interval), end_time)
        if window_start >= end_time:
            continue
        window_mask = (time_index >= window_start) & (time_index < window_end)
        if not window_mask.any():
            continue

        profile = np.nanmean(data_matrix[:, window_mask], axis=1)
        profiles.append(profile)
        rows.append(
            {
                "window_start": window_start,
                "window_end": window_end,
                "profile_center_time": window_start,
                "n_samples": int(window_mask.sum()),
                "all_nan_profile": bool(np.isnan(profile).all()),
            }
        )

    profile_matrix = (
        np.column_stack(profiles)
        if profiles
        else np.empty((data_matrix.shape[0], 0))
    )
    return profile_matrix, pd.DataFrame(rows)


def export_depth_time_csv(data_matrix, depth_values, time_values, csv_path):
    """Export a depth-by-time waterfall matrix using timestamp column names."""
    columns = {"measured_depth_ft": np.asarray(depth_values, dtype=float)}
    columns.update(
        {
            pd.Timestamp(time_value).strftime("%Y-%m-%d %H:%M:%S"): data_matrix[:, col_idx]
            for col_idx, time_value in enumerate(pd.DatetimeIndex(time_values))
        }
    )
    waterfall_df = pd.DataFrame(columns)
    waterfall_df.to_csv(csv_path, index=False)
    return waterfall_df


def export_profile_csv(profile_matrix, profile_metadata, depth_values, data_csv, metadata_csv):
    """Export the exact plotted profiles and their observation-style window metadata."""
    profile_df = pd.DataFrame({"measured_depth_ft": depth_values})
    for col_idx, row in profile_metadata.iterrows():
        column_name = pd.Timestamp(row["profile_center_time"]).strftime("%Y-%m-%d %H:%M:%S")
        profile_df[column_name] = profile_matrix[:, col_idx]
    profile_df.to_csv(data_csv, index=False)
    profile_metadata.to_csv(metadata_csv, index=False)
    return profile_df


def get_profile_abs_scale(profile_matrix, multiplier=1.0):
    if profile_matrix.size == 0:
        return np.nan, np.nan
    profile_p95 = np.nanpercentile(np.abs(profile_matrix), 95)
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        profile_p95 = np.nanmax(np.abs(profile_matrix))
    if not np.isfinite(profile_p95) or profile_p95 == 0:
        return np.nan, np.nan
    return profile_p95, profile_p95 * multiplier


def overlay_profiles_on_time_axis(
    ax,
    profile_matrix,
    profile_metadata,
    depth_values,
    half_width_hours,
    scale_reference_abs,
    color="black",
):
    if profile_matrix.size == 0 or not np.isfinite(scale_reference_abs) or scale_reference_abs == 0:
        return

    scale_seconds_per_unit = half_width_hours * 3600.0 / scale_reference_abs
    for col_idx, row in profile_metadata.iterrows():
        profile = profile_matrix[:, col_idx]
        finite = np.isfinite(profile)
        if not finite.any():
            continue
        profile_time = pd.Timestamp(row["profile_center_time"])
        x_times = profile_time + pd.to_timedelta(
            profile[finite] * scale_seconds_per_unit,
            unit="s",
        )
        ax.plot(
            x_times,
            depth_values[finite],
            color=color,
            linewidth=0.8,
            alpha=0.9,
            zorder=5,
        )


def plot_sim_waterfall(ax, data_matrix, time_values, depth_values, cmap_name, color_scale, cbar_label, title):
    x0 = mdates.date2num(pd.Timestamp(time_values[0]).to_pydatetime())
    x1 = mdates.date2num(pd.Timestamp(time_values[-1]).to_pydatetime())
    image = ax.imshow(
        data_matrix,
        cmap=cmap_name,
        aspect="auto",
        vmin=color_scale[0],
        vmax=color_scale[1],
        extent=(x0, x1, depth_values[-1], depth_values[0]),
        interpolation="none",
    )
    ax.xaxis_date()
    ax.set_ylabel("Measured Depth (ft)")
    ax.set_ylim(sim_md_bottom, sim_md_top)
    ax.set_title(title)
    colorbar = plt.colorbar(image, ax=ax)
    colorbar.set_label(cbar_label)
    return image


# Restrict the DDM result to the same T1-T3 and 10200-10500 ft view as the observation.
sim_time_all = pd.DatetimeIndex(
    [T0 + timedelta(minutes=float(value)) for value in np.asarray(strain_data.taxis)]
)
sim_depth_all = get_ddm_depth_axis(strain_data)
sim_time_mask = (sim_time_all >= sim_plot_start) & (sim_time_all <= sim_plot_end)
sim_md_mask = (sim_depth_all >= sim_md_top) & (sim_depth_all <= sim_md_bottom)

SimTime_plot = sim_time_all[sim_time_mask]
SimDepth_plot = sim_depth_all[sim_md_mask]
SimRate_plot = (
    np.asarray(strain_rate_data.data, dtype=float)[sim_md_mask, :][:, sim_time_mask]
    * MODEL_RATE_TO_NANOSTRAIN_PER_S
)
SimDirectStrain_plot_mstrain = (
    np.asarray(strain_data.data, dtype=float)[sim_md_mask, :][:, sim_time_mask]
    * MODEL_STRAIN_TO_MILLISTRAIN
)

if SimTime_plot.empty:
    raise RuntimeError("No simulated time samples remain in the T1-T3 window.")
if SimDepth_plot.size == 0:
    raise RuntimeError("No simulated depth samples remain in the requested MD window.")

# Compute rate profiles exactly as in the observation workflow.
SimRate_profile_matrix, sim_rate_profile_metadata = compute_mean_profiles(
    SimRate_plot,
    SimTime_plot,
    sim_plot_start,
    sim_plot_end,
    sim_profile_interval,
)

# Use direct DDM strain, then reference it to the first 4-hour T1 mean profile.
SimDirectStrain_profile_raw, sim_strain_profile_metadata = compute_mean_profiles(
    SimDirectStrain_plot_mstrain,
    SimTime_plot,
    sim_plot_start,
    sim_plot_end,
    sim_profile_interval,
)
if SimDirectStrain_profile_raw.shape[1] > 0:
    sim_strain_reference_profile = SimDirectStrain_profile_raw[:, [0]]
else:
    sim_strain_reference_profile = np.zeros((len(SimDepth_plot), 1))

SimDirectStrain_profile_matrix = (
    SimDirectStrain_profile_raw - sim_strain_reference_profile
)
SimDirectStrain_ref_mstrain = (
    SimDirectStrain_plot_mstrain - sim_strain_reference_profile
)

sim_rate_profile_metadata["profile_units"] = "nanostrain/s"
sim_rate_profile_metadata["profile_source"] = "DDM centered time derivative of direct strain"
sim_rate_profile_metadata["overlay_half_width_hours"] = sim_rate_overlay_half_width_hours
sim_rate_profile_metadata["overlay_scale_multiplier"] = sim_profile_scale_multiplier

sim_strain_profile_metadata["profile_units"] = "millistrain"
sim_strain_profile_metadata["profile_source"] = "direct DDM strain"
sim_strain_profile_metadata["strain_reference"] = (
    "first 4-hour mean window starting at T1"
)
sim_strain_profile_metadata["strain_reference_time"] = sim_plot_start
sim_strain_profile_metadata["overlay_half_width_hours"] = sim_strain_overlay_half_width_hours
sim_strain_profile_metadata["overlay_scale_multiplier"] = sim_profile_scale_multiplier

sim_rate_p95, sim_rate_scale_reference = get_profile_abs_scale(
    SimRate_profile_matrix,
    multiplier=sim_profile_scale_multiplier,
)
sim_strain_p95, sim_strain_scale_reference = get_profile_abs_scale(
    SimDirectStrain_profile_matrix,
    multiplier=sim_profile_scale_multiplier,
)
sim_rate_profile_metadata["profile_p95_abs"] = sim_rate_p95
sim_rate_profile_metadata["overlay_scale_reference_abs"] = sim_rate_scale_reference
sim_strain_profile_metadata["profile_p95_abs"] = sim_strain_p95
sim_strain_profile_metadata["overlay_scale_reference_abs"] = sim_strain_scale_reference

sim_rate_waterfall_csv = sim_export_dir / f"sim_rate_waterfall_{sim_export_suffix}.csv"
sim_direct_strain_waterfall_csv = sim_export_dir / f"sim_direct_strain_waterfall_{sim_export_suffix}.csv"
sim_rate_profile_csv = sim_export_dir / f"sim_rate_4h_{sim_export_suffix}.csv"
sim_rate_profile_meta_csv = sim_export_dir / f"sim_rate_4h_{sim_export_suffix}_metadata.csv"
sim_strain_profile_csv = sim_export_dir / f"sim_direct_strain_4h_{sim_export_suffix}.csv"
sim_strain_profile_meta_csv = sim_export_dir / f"sim_direct_strain_4h_{sim_export_suffix}_metadata.csv"

export_depth_time_csv(
    SimRate_plot,
    SimDepth_plot,
    SimTime_plot,
    sim_rate_waterfall_csv,
)
export_depth_time_csv(
    SimDirectStrain_ref_mstrain,
    SimDepth_plot,
    SimTime_plot,
    sim_direct_strain_waterfall_csv,
)

export_profile_csv(
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimDepth_plot,
    sim_rate_profile_csv,
    sim_rate_profile_meta_csv,
)
export_profile_csv(
    SimDirectStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimDepth_plot,
    sim_strain_profile_csv,
    sim_strain_profile_meta_csv,
)

fig, (ax_rate_profile, ax_strain_profile) = plt.subplots(
    2,
    1,
    figsize=(16, 9),
    sharex=True,
    constrained_layout=True,
)

plot_sim_waterfall(
    ax_rate_profile,
    SimRate_plot,
    SimTime_plot,
    SimDepth_plot,
    "bwr",
    sim_rate_color_scale,
    "Strain rate (nanostrain/s)",
    "DDM Strain-rate Waterfall with Overlaid 4-hour Mean Profiles",
)
overlay_profiles_on_time_axis(
    ax_rate_profile,
    SimRate_profile_matrix,
    sim_rate_profile_metadata,
    SimDepth_plot,
    sim_rate_overlay_half_width_hours,
    sim_rate_scale_reference,
)

plot_sim_waterfall(
    ax_strain_profile,
    SimDirectStrain_ref_mstrain,
    SimTime_plot,
    SimDepth_plot,
    "seismic",
    sim_strain_color_scale,
    "Strain (millistrain)",
    "Direct DDM Strain Waterfall with Overlaid 4-hour Mean Profiles (T1 Referenced)",
)
overlay_profiles_on_time_axis(
    ax_strain_profile,
    SimDirectStrain_profile_matrix,
    sim_strain_profile_metadata,
    SimDepth_plot,
    sim_strain_overlay_half_width_hours,
    sim_strain_scale_reference,
)

for mark_time, mark_label in zip(sim_time_marks, sim_mark_labels):
    for ax in (ax_rate_profile, ax_strain_profile):
        ax.axvline(mark_time, color="green", linestyle="--", linewidth=1.8, alpha=0.95)
        ax.text(
            mark_time,
            1.01,
            mark_label,
            color="green",
            fontsize=12,
            fontweight="bold",
            ha="center",
            va="bottom",
            transform=ax.get_xaxis_transform(),
            clip_on=False,
        )

ax_rate_profile.set_xlim(sim_plot_start, sim_plot_end)
ax_strain_profile.set_xlim(sim_plot_start, sim_plot_end)
ax_strain_profile.set_xlabel("Time [UTC-7]")
ax_strain_profile.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
plt.show()

print("Created direct-DDM strain-rate and strain waterfalls with observation-style 4-hour profiles.")
print(f"First profile time (T1): {sim_rate_profile_metadata.iloc[0]['profile_center_time']}")
print(f"Plotted/exported {SimRate_profile_matrix.shape[1]} strain-rate profiles.")
print(f"Plotted/exported {SimDirectStrain_profile_matrix.shape[1]} direct-strain profiles.")
print(f"Strain-rate waterfall CSV: {sim_rate_waterfall_csv}")
print(f"Direct-strain waterfall CSV: {sim_direct_strain_waterfall_csv}")
print(f"Strain-rate profile CSV: {sim_rate_profile_csv}")
print(f"Strain-rate metadata CSV: {sim_rate_profile_meta_csv}")
print(f"Direct-strain profile CSV: {sim_strain_profile_csv}")
print(f"Direct-strain metadata CSV: {sim_strain_profile_meta_csv}")


# One fault for tensile, one fault for shear

In [ ]:
# Two-fracture model: fault 1 carries tensile opening from T1-T3; fault 2 carries shear only from T2-T3.
# Fault 1 uses strike -0.8 deg. Fault 2 uses strike -0.3 deg.
# Both faults use the same center, dip, and height, with independently configurable lengths.

import copy
import os
from datetime import datetime, timedelta
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


# === 1. Monitoring-well geometry ===
base_path = r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Aleksei\Data\Well Geometries"
df_gold = pd.read_csv(f"{base_path}\\Gold_4_PB_Well_Geometry.csv")
control_points_G4 = df_gold[["x_gold", "y_gold", "z_gold"]].values
tvd = df_gold["TVDrkb"].values
md = df_gold["MD"].values

well_elem_N = len(control_points_G4) * 10
smooth_val = 30
smooth = smooth_val if smooth_val % 2 == 1 else smooth_val + 1
well = Well.set_well_by_points(control_points_G4, N=well_elem_N, smooth=smooth)
well.gauge_length = 30.48  # ft (10 m)


# === 2. Shared fracture geometry and displacement magnitudes ===
fault1_strike = -0.8  # current T1-T2 fracture geometry
fault2_strike = 0.6  # requested T2-T3 shear-fracture geometry
dip = 90
fault1_L = 3840
fault2_L = 3800
fracture_height = 4000
W_max = 0.008
S1_max = - 0.03

df_xyz = pd.read_csv(
    r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\input\pumping_windows_stage_with_coordinates_xyz.csv"
)
x = df_xyz["X"].values[5]
y = df_xyz["Y"].values[5]
z = df_xyz["Z"].values[5]


# === 3. Time axis ===
T0 = datetime(2025, 2, 24, 0, 0)
T1 = datetime(2025, 2, 24, 11, 0)
T2 = datetime(2025, 2, 28, 0, 0)
T3 = datetime(2025, 3, 3, 22, 0)


def mins(value):
    return (value - T0).total_seconds() / 60.0


t1 = mins(T1)
t2 = mins(T2)
t3 = mins(T3)

N = 144
delta_t = t3 / N
N1 = int(t1 // delta_t)
N2 = int((t2 - t1) // delta_t + 1)
N3 = int((t3 - t2 + 1) // delta_t + 1)

t_stage1 = np.linspace(0, t1 - delta_t, N1)
t_stage2 = np.linspace(t1, t2 - delta_t, N2)
t_stage3 = np.linspace(t2, t3, N3)
taxis = np.concatenate([t_stage1, t_stage2, t_stage3])
fault1_length = np.full_like(taxis, fault1_L)
fault2_length = np.full_like(taxis, fault2_L)
height = np.full_like(taxis, fracture_height)


# === 4. Two independent displacement histories ===
# Fault 1 only: zero opening before T1, then continuous tensile opening through T3.
# Preserve the original stage histories: 0 -> W_max during T1-T2 and W_max -> 5*W_max during T2-T3.
fault1_width_stage1 = np.zeros(N1)
fault1_width_stage2 = W_max * (np.linspace(0, 1, N2) ** 2)
fault1_width_stage3 = np.linspace(W_max, W_max * 3, N3 + 1)[1:]
fault1_width = np.concatenate(
    [fault1_width_stage1, fault1_width_stage2, fault1_width_stage3]
)
fault1_shear = np.zeros_like(taxis)

# Fault 2 only: no tensile opening at any time; shear starts at T2 and continues through T3.
fault2_width = np.zeros_like(taxis)
fault2_shear_stage3 = np.linspace(0, S1_max, N3)
fault2_shear = np.concatenate([np.zeros(N1 + N2), fault2_shear_stage3])

if not np.allclose(fault1_shear, 0.0):
    raise RuntimeError("Fault 1 must be tensile-only: nonzero shear was found.")
if not np.allclose(fault2_width, 0.0):
    raise RuntimeError("Fault 2 must be shear-only: nonzero tensile opening was found.")


if not (
    len(taxis)
    == len(fault1_width)
    == len(fault1_shear)
    == len(fault2_width)
    == len(fault2_shear)
):
    raise RuntimeError("Two-fracture histories do not match the shared time axis.")


def calculate_fracture_response(strike_value, length_history, width_history, shear_history):
    fracture = DynamicFracture.GlobalRectangularFracture()
    fracture.set_global_coors(strike_value, dip, x, y, z)
    fracture.define_LHW(
        taxis=taxis,
        length=length_history,
        height=height,
        width=width_history,
        S1=shear_history,
        S2=None,
    )
    fracture.set_monitor_wells(well)
    fracture.calculate()
    return fracture, fracture.gather_strain_data()[0]


fault1, fault1_strain_data = calculate_fracture_response(
    fault1_strike,
    fault1_length,
    fault1_width,
    fault1_shear,
)
fault2, fault2_strain_data = calculate_fracture_response(
    fault2_strike,
    fault2_length,
    fault2_width,
    fault2_shear,
)

if np.asarray(fault1_strain_data.data).shape != np.asarray(fault2_strain_data.data).shape:
    raise RuntimeError("Fault 1 and fault 2 strain arrays have different shapes.")
if not np.allclose(
    np.asarray(fault1_strain_data.taxis, dtype=float),
    np.asarray(fault2_strain_data.taxis, dtype=float),
):
    raise RuntimeError("Fault 1 and fault 2 time axes do not match.")

# Linear-elastic DDM superposition: total strain = fault 1 strain + fault 2 strain.
strain_data = copy.deepcopy(fault1_strain_data)
strain_data.data = (
    np.asarray(fault1_strain_data.data, dtype=float)
    + np.asarray(fault2_strain_data.data, dtype=float)
)


# === 5. Total strain rate and integrated-strain comparison ===
taxis_seconds = np.asarray(strain_data.taxis, dtype=float) * 60.0
strain_rate_data = copy.deepcopy(strain_data)
strain_rate_data.data = StrainRateUtil.centered_time_diff(
    np.asarray(strain_data.data, dtype=float),
    taxis_seconds,
)

strain_t1_ref_data = copy.deepcopy(strain_data)
t1_strain_idx = int(np.argmin(np.abs(np.asarray(strain_data.taxis, dtype=float) - t1)))
t1_strain_time = float(strain_data.taxis[t1_strain_idx])
strain_t1_ref_data.data = (
    np.asarray(strain_data.data, dtype=float)
    - np.asarray(strain_data.data, dtype=float)[:, [t1_strain_idx]]
)

integrated_strain_data = copy.deepcopy(strain_rate_data)
integrated_strain_data.data = StrainRateUtil.integrate_rate_trapezoidal(
    np.asarray(strain_rate_data.data, dtype=float),
    taxis_seconds,
)
integrated_strain_t1_ref_data = copy.deepcopy(integrated_strain_data)
t1_integrated_idx = int(
    np.argmin(np.abs(np.asarray(integrated_strain_data.taxis, dtype=float) - t1))
)
t1_integrated_time = float(integrated_strain_data.taxis[t1_integrated_idx])
integrated_strain_t1_ref_data.data = (
    np.asarray(integrated_strain_data.data, dtype=float)
    - np.asarray(integrated_strain_data.data, dtype=float)[:, [t1_integrated_idx]]
)

strain_integration_difference = (
    np.asarray(strain_t1_ref_data.data, dtype=float)
    - np.asarray(integrated_strain_t1_ref_data.data, dtype=float)
)
max_abs_strain_integration_difference = float(
    np.nanmax(np.abs(strain_integration_difference))
)


# === 6. Draw the two fracture geometries separately ===
fig_fault1 = fault1.draw(tensile=0.02)
fig_fault1.update_layout(
    title=f"Fault 1: T1-T3 tensile-only fracture, strike={fault1_strike:.1f} deg"
)
fig_fault1.show()

fig_fault2 = fault2.draw(tensile=0.02)
fig_fault2.update_layout(
    title=f"Fault 2: T2-T3 shear-only fracture, strike={fault2_strike:.1f} deg"
)
fig_fault2.show()


# === 7. Total strain-rate/direct-strain/integrated-strain waterfalls ===
tick_minutes = np.arange(0, mins(datetime(2025, 3, 4, 0, 0)) + 1, 24 * 60)
tick_labels = [
    (T0 + timedelta(minutes=int(value))).strftime("%m/%d\n%H:%M")
    for value in tick_minutes
]

fig_compare, (ax_rate, ax_direct, ax_integrated) = plt.subplots(
    3,
    1,
    figsize=(12, 12),
    sharex=True,
    constrained_layout=True,
)

plt.sca(ax_rate)
strain_rate_data.plot_waterfall()
rate_mappable = ax_rate.images[-1] if ax_rate.images else ax_rate.collections[-1]
rate_mappable.set_clim(-0.3e-9, 0.3e-9)
rate_cbar = fig_compare.colorbar(rate_mappable, ax=ax_rate)
rate_cbar.set_label("Total strain rate (nanostrain/s)")
rate_cbar.formatter = FuncFormatter(lambda value, _: f"{value * 1e9:g}")
rate_cbar.update_ticks()
ax_rate.set_title("Two-fault total strain-rate waterfall")

plt.sca(ax_direct)
strain_t1_ref_data.plot_waterfall()
direct_mappable = ax_direct.images[-1] if ax_direct.images else ax_direct.collections[-1]
direct_mappable.set_clim(-0.1e-3, 0.1e-3)
direct_cbar = fig_compare.colorbar(direct_mappable, ax=ax_direct)
direct_cbar.set_label("Total direct DDM strain (millistrain)")
direct_cbar.formatter = FuncFormatter(lambda value, _: f"{value * 1e3:g}")
direct_cbar.update_ticks()
ax_direct.set_title("Two-fault total direct DDM strain (T1 referenced)")

plt.sca(ax_integrated)
integrated_strain_t1_ref_data.plot_waterfall()
integrated_mappable = (
    ax_integrated.images[-1]
    if ax_integrated.images
    else ax_integrated.collections[-1]
)
integrated_mappable.set_clim(-0.1e-3, 0.1e-3)
integrated_cbar = fig_compare.colorbar(integrated_mappable, ax=ax_integrated)
integrated_cbar.set_label("Total strain integrated from rate (millistrain)")
integrated_cbar.formatter = FuncFormatter(lambda value, _: f"{value * 1e3:g}")
integrated_cbar.update_ticks()
ax_integrated.set_title("Two-fault strain integrated from strain rate (T1 referenced)")

for axis in (ax_rate, ax_direct, ax_integrated):
    axis.set_ylabel("Measured Depth (ft)")
    axis.set_ylim(10500, 10200)
    axis.set_xlim(t1, t3)
    axis.set_xticks(tick_minutes)
    axis.set_xticklabels(tick_labels)
    for mark_value, mark_label in zip((t1, t2, t3), ("T1", "T2", "T3")):
        axis.axvline(mark_value, color="green", linestyle="--", linewidth=1.4, label=mark_label)
    axis.legend(loc="best")

ax_integrated.set_xlabel("Time")
plt.show()


# === 8. Observation-style T1-T3 / 10200-10500 ft profiles and exports ===
plot_start = pd.Timestamp(T1)
plot_end = pd.Timestamp(T3)
plot_md_top = 10200
plot_md_bottom = 10500
profile_interval = "4h"
rate_overlay_half_width_hours = 28.0
strain_overlay_half_width_hours = 28.0
profile_scale_multiplier = 10.0
rate_color_scale = (-0.3, 0.3)
strain_color_scale = (-0.1, 0.1)

export_dir = Path(
    r"D:\OneDrive - Colorado School of Mines\Research\Fervo\Pengchao\code\profile_exports\sim_two_faults"
)
export_dir.mkdir(parents=True, exist_ok=True)
export_suffix = (
    f"{plot_start:%Y%m%d_%H%M}_to_{plot_end:%Y%m%d_%H%M}_"
    f"{plot_md_top}_{plot_md_bottom}ft_{profile_interval}_mean_T1_ref"
)


def get_depth_axis(data_obj):
    for name in ("daxis", "depth", "md", "yaxis", "xaxis"):
        if hasattr(data_obj, name):
            values = getattr(data_obj, name)
            if values is not None:
                return np.asarray(values, dtype=float)
    return np.arange(np.asarray(data_obj.data).shape[0], dtype=float)


def compute_mean_profiles(data_matrix, time_index, start_time, end_time, interval):
    time_index = pd.DatetimeIndex(time_index)
    profiles = []
    rows = []
    for window_start in pd.date_range(start_time, end_time, freq=interval):
        window_end = min(window_start + pd.Timedelta(interval), end_time)
        if window_start >= end_time:
            continue
        mask = (time_index >= window_start) & (time_index < window_end)
        if not mask.any():
            continue
        profiles.append(np.nanmean(data_matrix[:, mask], axis=1))
        rows.append(
            {
                "window_start": window_start,
                "window_end": window_end,
                "profile_center_time": window_start,
                "n_samples": int(mask.sum()),
                "all_nan_profile": False,
            }
        )
    matrix = np.column_stack(profiles) if profiles else np.empty((data_matrix.shape[0], 0))
    return matrix, pd.DataFrame(rows)


def get_profile_abs_scale(profile_matrix, multiplier):
    if profile_matrix.size == 0:
        return np.nan, np.nan
    p95 = np.nanpercentile(np.abs(profile_matrix), 95)
    if not np.isfinite(p95) or p95 == 0:
        p95 = np.nanmax(np.abs(profile_matrix))
    if not np.isfinite(p95) or p95 == 0:
        return np.nan, np.nan
    return p95, p95 * multiplier


def overlay_profiles(ax, matrix, metadata, depths, half_width_hours, scale_reference):
    if matrix.size == 0 or not np.isfinite(scale_reference) or scale_reference == 0:
        return
    seconds_per_unit = half_width_hours * 3600.0 / scale_reference
    for column_index, row in metadata.iterrows():
        profile = matrix[:, column_index]
        finite = np.isfinite(profile)
        profile_time = pd.Timestamp(row["profile_center_time"])
        x_values = profile_time + pd.to_timedelta(
            profile[finite] * seconds_per_unit,
            unit="s",
        )
        ax.plot(x_values, depths[finite], color="black", linewidth=0.8, alpha=0.9, zorder=5)


def export_depth_time_csv(data_matrix, depth_values, time_values, csv_path):
    columns = {"measured_depth_ft": np.asarray(depth_values, dtype=float)}
    columns.update(
        {
            pd.Timestamp(time_value).strftime("%Y-%m-%d %H:%M:%S"): data_matrix[:, index]
            for index, time_value in enumerate(pd.DatetimeIndex(time_values))
        }
    )
    pd.DataFrame(columns).to_csv(csv_path, index=False)


def export_profile_csv(matrix, metadata, depths, data_csv, metadata_csv):
    columns = {"measured_depth_ft": np.asarray(depths, dtype=float)}
    columns.update(
        {
            pd.Timestamp(row["profile_center_time"]).strftime("%Y-%m-%d %H:%M:%S"): matrix[:, index]
            for index, row in metadata.iterrows()
        }
    )
    pd.DataFrame(columns).to_csv(data_csv, index=False)
    metadata.to_csv(metadata_csv, index=False)


all_times = pd.DatetimeIndex(
    [T0 + timedelta(minutes=float(value)) for value in np.asarray(strain_data.taxis)]
)
all_depths = get_depth_axis(strain_data)
time_mask = (all_times >= plot_start) & (all_times <= plot_end)
depth_mask = (all_depths >= plot_md_top) & (all_depths <= plot_md_bottom)

Time_plot = all_times[time_mask]
Depth_plot = all_depths[depth_mask]
Rate_plot = (
    np.asarray(strain_rate_data.data, dtype=float)[depth_mask, :][:, time_mask] * 1e9
)
DirectStrain_plot_raw = (
    np.asarray(strain_data.data, dtype=float)[depth_mask, :][:, time_mask] * 1e3
)

Rate_profile_matrix, rate_profile_metadata = compute_mean_profiles(
    Rate_plot,
    Time_plot,
    plot_start,
    plot_end,
    profile_interval,
)
DirectStrain_profile_raw, strain_profile_metadata = compute_mean_profiles(
    DirectStrain_plot_raw,
    Time_plot,
    plot_start,
    plot_end,
    profile_interval,
)

strain_reference_profile = (
    DirectStrain_profile_raw[:, [0]]
    if DirectStrain_profile_raw.shape[1] > 0
    else np.zeros((len(Depth_plot), 1))
)
DirectStrain_profile_matrix = DirectStrain_profile_raw - strain_reference_profile
DirectStrain_plot = DirectStrain_plot_raw - strain_reference_profile

rate_profile_p95, rate_profile_scale = get_profile_abs_scale(
    Rate_profile_matrix,
    profile_scale_multiplier,
)
strain_profile_p95, strain_profile_scale = get_profile_abs_scale(
    DirectStrain_profile_matrix,
    profile_scale_multiplier,
)

rate_profile_metadata["profile_units"] = "nanostrain/s"
rate_profile_metadata["profile_source"] = "two-fault total strain rate"
rate_profile_metadata["overlay_half_width_hours"] = rate_overlay_half_width_hours
rate_profile_metadata["overlay_scale_multiplier"] = profile_scale_multiplier
rate_profile_metadata["profile_p95_abs"] = rate_profile_p95
rate_profile_metadata["overlay_scale_reference_abs"] = rate_profile_scale

strain_profile_metadata["profile_units"] = "millistrain"
strain_profile_metadata["profile_source"] = "two-fault total direct DDM strain"
strain_profile_metadata["strain_reference"] = "first 4-hour mean window starting at T1"
strain_profile_metadata["strain_reference_time"] = plot_start
strain_profile_metadata["overlay_half_width_hours"] = strain_overlay_half_width_hours
strain_profile_metadata["overlay_scale_multiplier"] = profile_scale_multiplier
strain_profile_metadata["profile_p95_abs"] = strain_profile_p95
strain_profile_metadata["overlay_scale_reference_abs"] = strain_profile_scale

rate_waterfall_csv = export_dir / f"two_fault_rate_waterfall_{export_suffix}.csv"
strain_waterfall_csv = export_dir / f"two_fault_direct_strain_waterfall_{export_suffix}.csv"
rate_profile_csv = export_dir / f"two_fault_rate_4h_{export_suffix}.csv"
rate_profile_meta_csv = export_dir / f"two_fault_rate_4h_{export_suffix}_metadata.csv"
strain_profile_csv = export_dir / f"two_fault_direct_strain_4h_{export_suffix}.csv"
strain_profile_meta_csv = export_dir / f"two_fault_direct_strain_4h_{export_suffix}_metadata.csv"
history_csv = export_dir / f"two_fault_histories_{export_suffix}.csv"

export_depth_time_csv(Rate_plot, Depth_plot, Time_plot, rate_waterfall_csv)
export_depth_time_csv(DirectStrain_plot, Depth_plot, Time_plot, strain_waterfall_csv)
export_profile_csv(
    Rate_profile_matrix,
    rate_profile_metadata,
    Depth_plot,
    rate_profile_csv,
    rate_profile_meta_csv,
)
export_profile_csv(
    DirectStrain_profile_matrix,
    strain_profile_metadata,
    Depth_plot,
    strain_profile_csv,
    strain_profile_meta_csv,
)

pd.DataFrame(
    {
        "time_minutes_since_T0": taxis,
        "time": [T0 + timedelta(minutes=float(value)) for value in taxis],
        "fault1_strike_deg": fault1_strike,
        "fault1_width_ft": fault1_width,
        "fault1_shear_ft": fault1_shear,
        "fault2_strike_deg": fault2_strike,
        "fault2_width_ft": fault2_width,
        "fault2_shear_ft": fault2_shear,
    }
).to_csv(history_csv, index=False)


def plot_matrix(ax, matrix, times, depths, cmap, limits, label, title):
    image = ax.imshow(
        matrix,
        cmap=cmap,
        aspect="auto",
        vmin=limits[0],
        vmax=limits[1],
        extent=(
            mdates.date2num(times[0].to_pydatetime()),
            mdates.date2num(times[-1].to_pydatetime()),
            depths[-1],
            depths[0],
        ),
        interpolation="none",
    )
    ax.xaxis_date()
    ax.set_ylim(plot_md_bottom, plot_md_top)
    ax.set_ylabel("Measured Depth (ft)")
    ax.set_title(title)
    colorbar = plt.colorbar(image, ax=ax)
    colorbar.set_label(label)


fig_profiles, (ax_rate_profiles, ax_strain_profiles) = plt.subplots(
    2,
    1,
    figsize=(16, 9),
    sharex=True,
    constrained_layout=True,
)
plot_matrix(
    ax_rate_profiles,
    Rate_plot,
    Time_plot,
    Depth_plot,
    "bwr",
    rate_color_scale,
    "Strain rate (nanostrain/s)",
    "Two-fault total strain-rate waterfall with 4-hour mean profiles",
)
overlay_profiles(
    ax_rate_profiles,
    Rate_profile_matrix,
    rate_profile_metadata,
    Depth_plot,
    rate_overlay_half_width_hours,
    rate_profile_scale,
)

plot_matrix(
    ax_strain_profiles,
    DirectStrain_plot,
    Time_plot,
    Depth_plot,
    "seismic",
    strain_color_scale,
    "Strain (millistrain)",
    "Two-fault total direct DDM strain with 4-hour mean profiles (T1 referenced)",
)
overlay_profiles(
    ax_strain_profiles,
    DirectStrain_profile_matrix,
    strain_profile_metadata,
    Depth_plot,
    strain_overlay_half_width_hours,
    strain_profile_scale,
)

for axis in (ax_rate_profiles, ax_strain_profiles):
    for mark_time, mark_label in zip((T1, T2, T3), ("T1", "T2", "T3")):
        axis.axvline(mark_time, color="green", linestyle="--", linewidth=1.5)
        axis.text(
            mark_time,
            1.01,
            mark_label,
            color="green",
            fontweight="bold",
            ha="center",
            va="bottom",
            transform=axis.get_xaxis_transform(),
            clip_on=False,
        )
    axis.set_xlim(plot_start, plot_end)

ax_strain_profiles.set_xlabel("Time [UTC-7]")
ax_strain_profiles.xaxis.set_major_formatter(mdates.DateFormatter("%m/%d\n%H:%M"))
plt.show()


# === 9. Summary ===
fault1_abs_max = float(np.nanmax(np.abs(fault1_strain_data.data)))
fault2_abs_max = float(np.nanmax(np.abs(fault2_strain_data.data)))
total_abs_max = float(np.nanmax(np.abs(strain_data.data)))

print("Created two fractures and superposed their linear-elastic DDM responses.")
print(f"Fault 1: strike={fault1_strike:.1f} deg; tensile opening runs from T1 through T3; shear is zero.")
print(f"Fault 2: strike={fault2_strike:.1f} deg; width is zero; shear runs from T2 through T3.")
print(f"Fault 1 max |strain|: {fault1_abs_max:.6e}")
print(f"Fault 2 max |strain|: {fault2_abs_max:.6e}")
print(f"Combined max |strain|: {total_abs_max:.6e}")
print(f"Direct-vs-integrated max difference: {max_abs_strain_integration_difference:.6e}")
print(f"First 4-hour profile time: {rate_profile_metadata.iloc[0]['profile_center_time']}")
print(f"Exported {Rate_profile_matrix.shape[1]} rate and {DirectStrain_profile_matrix.shape[1]} strain profiles.")
print(f"Rate waterfall CSV: {rate_waterfall_csv}")
print(f"Direct-strain waterfall CSV: {strain_waterfall_csv}")
print(f"Rate profile CSV: {rate_profile_csv}")
print(f"Rate metadata CSV: {rate_profile_meta_csv}")
print(f"Direct-strain profile CSV: {strain_profile_csv}")
print(f"Direct-strain metadata CSV: {strain_profile_meta_csv}")
print(f"Two-fracture history CSV: {history_csv}")
